# <font color="#418FDE" size="6.5" uppercase>**Responsible Model Use**</font>

>Last update: 20260710.
    
By the end of this Lecture, you will be able to:
- Interpret model behavior using coefficients, rules, errors, plots, and feature sensitivity checks. 
- Evaluate practical limitations such as overfitting, imbalance, bias, data drift, and explainability. 
- Communicate a complete mechanical engineering AI project with evidence-based recommendations. 


## **1. Model Interpretation**

### **1.1. Coefficient Interpretation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_08/Lecture_B/image_01_01.jpg?v=1783735826" width="250">



>* Coefficients show feature direction and influence
>* Interpret them within data and model limits

>* Interpret coefficient signs and sizes with scale
>* Investigate weak or unexpected coefficients carefully

>* Separate prediction signals from causal claims
>* Validate coefficients with engineering evidence



In [ ]:
#@title Python Code - Coefficient Interpretation

# Coefficients connect features with predicted engineering outcomes.
# Scaling helps compare coefficient sizes responsibly.
# Sensitivity checks show practical prediction changes.

# Import numerical, table, and plotting tools.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create deterministic pump operating data.
rng = np.random.default_rng(7)
n = 40
flow_gpm = rng.uniform(80, 220, n)

# Add efficiency and pressure features.
efficiency = rng.uniform(0.68, 0.92, n)
pressure_psi = rng.uniform(35, 95, n)
noise = rng.normal(0, 1.8, n)

# Build energy from known relationships.
energy_kw = 8 + 0.055 * flow_gpm
energy_kw += 0.030 * pressure_psi
energy_kw += -18.0 * efficiency + noise

# Store data in a compact table.
data = pd.DataFrame({
    "flow_gpm": flow_gpm,
    "pressure_psi": pressure_psi,
    "efficiency": efficiency,
    "energy_kw": energy_kw,
})

# Validate the small teaching dataset.
assert data.shape == (40, 4)
assert data.notna().all().all()

# Prepare standardized features for fair comparison.
features = ["flow_gpm", "pressure_psi", "efficiency"]
X = data[features].to_numpy()
y = data["energy_kw"].to_numpy()

# Standardize features before fitting coefficients.
means = X.mean(axis=0)
stds = X.std(axis=0, ddof=0)
X_scaled = (X - means) / stds

# Add intercept column for linear regression.
ones = np.ones((n, 1))
X_design = np.column_stack([ones, X_scaled])
coef = np.linalg.lstsq(X_design, y, rcond=None)[0]

# Convert coefficients into a readable table.
coef_table = pd.DataFrame({
    "feature": ["intercept"] + features,
    "coefficient": np.round(coef, 3),
})

# Print concise interpretation guidance.
print("Standardized linear coefficients for pump energy:")
print(coef_table.to_string(index=False))
print("Positive means higher predicted energy, holding others steady.")
print("Negative efficiency coefficient matches engineering intuition.")

# Run a simple one-feature sensitivity check.
baseline = X.mean(axis=0)
low_high = baseline.copy(), baseline.copy()
low_high[0][0] -= X[:, 0].std()
low_high[1][0] += X[:, 0].std()

# Predict energy at lower and higher flow.
scaled_cases = (np.vstack(low_high) - means) / stds
case_design = np.column_stack([np.ones(2), scaled_cases])
predictions = case_design @ coef

# Print the practical sensitivity result.
change = predictions[1] - predictions[0]
print(f"One standard deviation flow increase changes energy by {change:.2f} kW.")

# Plot coefficients for visual interpretation.
plt.figure(figsize=(7, 4))
colors = ["steelblue" if value >= 0 else "tomato" for value in coef[1:]]
plt.bar(features, coef[1:], color=colors)
plt.axhline(0, color="black", linewidth=1)
plt.ylabel("Standardized coefficient")
plt.title("Coefficient interpretation for pump energy model")
plt.tight_layout()
plt.show()



### **1.2. Rule Based Insights**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_08/Lecture_B/image_01_02.jpg?v=1783735828" width="250">



>* Rules turn predictions into inspectable conditions
>* Engineering context checks whether patterns make sense

>* Rules connect model patterns to expertise
>* Validate rules against real system evidence

>* Treat rules as testable evidence, not commands
>* Support recommendations with analysis and engineering judgment



In [ ]:
#@title Python Code - Rule Based Insights

# Rule insights translate predictions into inspectable engineering conditions.
# This example uses simple pump maintenance thresholds.
# We compare rules with observed failure outcomes.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create a tiny deterministic maintenance dataset.
pump_data = pd.DataFrame({
    "vibration_mm_s": [2.1, 3.4, 5.8, 6.2, 4.9, 7.1, 3.0, 6.8],
    "temperature_c": [58, 64, 82, 88, 76, 91, 61, 85],
    "hours": [900, 1400, 2600, 3100, 2200, 3600, 1200, 2900],
    "failed": [0, 0, 1, 1, 0, 1, 0, 1]
})

# Validate the dataset before applying rules.
required_columns = {"vibration_mm_s", "temperature_c", "hours", "failed"}
assert required_columns.issubset(pump_data.columns), "Missing required columns."
assert len(pump_data) >= 4, "Dataset is too small."

# Define an interpretable rule from engineering thresholds.
high_vibration = pump_data["vibration_mm_s"] >= 5.5
high_temperature = pump_data["temperature_c"] >= 80
long_service = pump_data["hours"] >= 2500

# Combine conditions into one rule prediction.
pump_data["rule_risk"] = high_vibration & high_temperature & long_service
pump_data["rule_prediction"] = pump_data["rule_risk"].astype(int)

# Summarize rule performance without printing full data.
correct = pump_data["rule_prediction"].eq(pump_data["failed"]).sum()
accuracy = correct / len(pump_data)
covered = int(pump_data["rule_risk"].sum())

# Print concise interpretation lines for learners.
print("Rule: high vibration, high temperature, and long service imply risk.")
print(f"Rule flagged {covered} of {len(pump_data)} pumps as high risk.")
print(f"Rule accuracy on this tiny example: {accuracy:.0%}.")
print("Insight: the rule is understandable, but needs broader validation.")

# Plot failures and rule flagged cases together.
colors = np.where(pump_data["rule_risk"], "crimson", "steelblue")
markers = np.where(pump_data["failed"].eq(1), "x", "o")
fig, ax = plt.subplots(figsize=(7, 4))

# Draw each point with its observed outcome marker.
for row_index, row in pump_data.iterrows():
    ax.scatter(row["vibration_mm_s"], row["temperature_c"],
               color=colors[row_index], marker=markers[row_index], s=90)

# Add threshold lines that define the rule.
ax.axvline(5.5, color="gray", linestyle="--", linewidth=1)
ax.axhline(80, color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("Vibration amplitude (mm/s)")
ax.set_ylabel("Bearing temperature (°C)")
ax.set_title("Rule based insight for pump failure risk")
plt.tight_layout()
plt.show()



### **1.3. Error Pattern Analysis**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_08/Lecture_B/image_01_03.jpg?v=1783735824" width="250">



>* Look beyond accuracy to find hidden mistakes
>* Use errors to understand model limits

>* Group errors by type and operating conditions
>* Use plots and engineering judgment together

>* Match error risks to engineering decisions
>* Communicate limits, drift, and needed updates



In [ ]:
#@title Python Code - Error Pattern Analysis

# Analyze model errors by operating condition.
# Use synthetic pump vibration data.
# Compare residuals across speed ranges.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set a deterministic random seed.
rng = np.random.default_rng(42)

# Create small operating-condition data.
n = 80
speed_rpm = rng.uniform(900, 3600, n)
load_percent = rng.uniform(30, 95, n)

# Simulate true vibration in millimeters.
noise = rng.normal(0, 0.18, n)
true_vibration = 0.002 * speed_rpm + 0.018 * load_percent + noise

# Simulate a biased simple model.
predicted_vibration = 0.0017 * speed_rpm + 0.020 * load_percent + 0.35

# Store predictions and residual errors.
data = pd.DataFrame({
    "speed_rpm": speed_rpm,
    "load_percent": load_percent,
    "actual_mm_s": true_vibration,
    "predicted_mm_s": predicted_vibration,
})

# Compute signed and absolute errors.
data["residual"] = data["actual_mm_s"] - data["predicted_mm_s"]
data["absolute_error"] = data["residual"].abs()

# Validate the table before analysis.
assert len(data) == n and data.notna().all().all()

# Group errors by speed range.
data["speed_group"] = pd.cut(
    data["speed_rpm"],
    bins=[0, 1800, 2700, 4000],
    labels=["low", "medium", "high"],
)

# Summarize error patterns compactly.
summary = data.groupby("speed_group", observed=True).agg(
    mean_residual=("residual", "mean"),
    mean_abs_error=("absolute_error", "mean"),
    cases=("residual", "size"),
)

# Identify the weakest operating region.
worst_group = summary["mean_abs_error"].idxmax()
worst_error = summary.loc[worst_group, "mean_abs_error"]

# Print concise teaching results.
print("Error pattern analysis for pump vibration model")
print(summary.round(3).to_string())
print(f"Largest average error occurs at {worst_group} speed.")
print(f"Mean absolute error there is {worst_error:.3f} mm/s.")
print("Positive residual means the model underpredicted vibration.")

# Plot residuals against speed.
plt.figure(figsize=(7, 4))
plt.axhline(0, color="black", linewidth=1)
plt.scatter(data["speed_rpm"], data["residual"], alpha=0.75)
plt.xlabel("Pump speed, rpm")
plt.ylabel("Residual, actual minus predicted mm/s")
plt.title("Residual plot reveals speed-related error patterns")
plt.tight_layout()
plt.show()



## **2. Practical Model Risks**

### **2.1. Data Drift**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_08/Lecture_B/image_02_01.jpg?v=1783735830" width="250">



>* Deployed data can change over time
>* Confident predictions may become unreliable

>* Input and concept drift change model reliability
>* Monitor models beyond initial validation

>* Monitor models as living engineering systems
>* Respond with retraining, controls, or review



In [ ]:
#@title Python Code - Data Drift

# Data drift can weaken deployed engineering models.
# This example compares development and operating data.
# A simple rule shows changing prediction error.

# Import numerical and plotting libraries.
import numpy as np
import matplotlib.pyplot as plt

# Make the random example repeatable.
rng = np.random.default_rng(42)
sample_count = 80
assert sample_count > 20

# Simulate development vibration in inches per second.
dev_vibration = rng.normal(0.18, 0.03, sample_count)
dev_noise = rng.normal(0.0, 0.4, sample_count)
dev_rul = 120 - 260 * dev_vibration + dev_noise

# Simulate later operation after wear increases vibration.
ops_vibration = rng.normal(0.26, 0.04, sample_count)
ops_noise = rng.normal(0.0, 0.4, sample_count)
ops_rul = 120 - 310 * ops_vibration + ops_noise

# Fit a tiny linear model using formulas.
x_mean = dev_vibration.mean()
y_mean = dev_rul.mean()
slope = np.sum((dev_vibration - x_mean) * (dev_rul - y_mean))
slope = slope / np.sum((dev_vibration - x_mean) ** 2)

# Compute intercept and predictions.
intercept = y_mean - slope * x_mean
dev_pred = intercept + slope * dev_vibration
ops_pred = intercept + slope * ops_vibration

# Measure average absolute prediction errors.
dev_mae = np.mean(np.abs(dev_rul - dev_pred))
ops_mae = np.mean(np.abs(ops_rul - ops_pred))
drift_size = ops_vibration.mean() - dev_vibration.mean()
error_ratio = ops_mae / max(dev_mae, 1e-9)

# Print a compact engineering interpretation.
print(f"Development mean vibration: {dev_vibration.mean():.3f} in/s")
print(f"Operating mean vibration:   {ops_vibration.mean():.3f} in/s")
print(f"Input drift size:           {drift_size:.3f} in/s")
print(f"Development MAE:            {dev_mae:.2f} hours")
print(f"Operating MAE:              {ops_mae:.2f} hours")
print(f"Error increase factor:      {error_ratio:.1f}x")

# Plot both feature distributions for visual monitoring.
plt.figure(figsize=(7, 4))
plt.hist(dev_vibration, bins=12, alpha=0.65, label="development")
plt.hist(ops_vibration, bins=12, alpha=0.65, label="operation")

# Label the drift monitoring plot clearly.
plt.xlabel("Pump vibration, inches per second")
plt.ylabel("Number of observations")
plt.title("Data drift: operating vibration shifted upward")
plt.legend()

# Display the single teaching plot.
plt.tight_layout()
plt.show()



### **2.2. Bias and Imbalance**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_08/Lecture_B/image_02_02.jpg?v=1783735834" width="250">



>* Imbalanced data can hide rare critical failures
>* Biased data limits reliability across conditions

>* Check performance across engineering conditions
>* Watch rare failures and biased recommendations

>* Check subgroup errors and missing conditions
>* Use safeguards and communicate limits clearly



In [ ]:
#@title Python Code - Bias and Imbalance

# Bias can hide rare failures.
# Imbalance can inflate accuracy.
# Group checks reveal model risk.

# Import small scientific plotting libraries.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create deterministic synthetic maintenance data.
rng = np.random.default_rng(42)
n_normal = 180
n_fault = 20

# Simulate vibration in millimeters per second.
normal_vib = rng.normal(2.0, 0.45, n_normal)
fault_vib = rng.normal(4.2, 0.70, n_fault)
vibration = np.concatenate([normal_vib, fault_vib])

# Store true labels and machine groups.
actual = np.array([0] * n_normal + [1] * n_fault)
groups = np.array(["Pump A"] * 120 + ["Pump B"] * 80)
threshold = 3.6

# Predict faults using one simple engineering rule.
predicted = (vibration > threshold).astype(int)
data = pd.DataFrame({"group": groups, "actual": actual, "predicted": predicted})

# Define compact metric calculations.
def metrics(frame):
    total = len(frame)
    correct = (frame["actual"] == frame["predicted"]).sum()
    faults = frame["actual"].sum()
    caught = ((frame["actual"] == 1) & (frame["predicted"] == 1)).sum()

    accuracy = correct / total
    recall = caught / max(faults, 1)
    return accuracy, recall, faults

# Compare overall and subgroup performance.
overall_acc, overall_rec, overall_faults = metrics(data)
pump_a_acc, pump_a_rec, pump_a_faults = metrics(data[data["group"] == "Pump A"])
pump_b_acc, pump_b_rec, pump_b_faults = metrics(data[data["group"] == "Pump B"])

# Print only key teaching results.
print(f"Overall accuracy: {overall_acc:.2f}")
print(f"Overall fault recall: {overall_rec:.2f}")
print(f"Pump A faults in data: {pump_a_faults}")
print(f"Pump B faults in data: {pump_b_faults}")
print(f"Pump A fault recall: {pump_a_rec:.2f}")
print(f"Pump B fault recall: {pump_b_rec:.2f}")

# Plot subgroup recall to expose imbalance.
labels = ["Pump A", "Pump B"]
recalls = [pump_a_rec, pump_b_rec]
plt.figure(figsize=(6, 4))
plt.bar(labels, recalls, color=["steelblue", "orange"])

# Add readable plot labels.
plt.ylim(0, 1.05)
plt.ylabel("Fault recall")
plt.title("Rare fault detection by machine group")
plt.grid(axis="y", alpha=0.3)

# Show the single teaching plot.
plt.show()



### **2.3. Bias and Drift**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_08/Lecture_B/image_02_03.jpg?v=1783735836" width="250">



>* Bias causes uneven model reliability
>* Drift exposes weaknesses in new conditions

>* Changing conditions can shift model errors
>* Drift can cause unequal real-world impacts

>* Monitor overall and subgroup model performance
>* Update controls as bias and drift appear



In [ ]:
#@title Python Code - Bias and Drift

# Bias and drift affect deployed engineering models.
# This example uses simple built-in calculations.
# We compare average and subgroup errors.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create deterministic operating data.
rng = np.random.default_rng(7)
n_each = 40

# Simulate newer and older machines.
ages = np.r_[rng.normal(2, 0.4, n_each), rng.normal(12, 1.0, n_each)]
groups = np.array(["new"] * n_each + ["old"] * n_each)

# Simulate vibration in millimeters per second.
vibration = 0.8 + 0.18 * ages + rng.normal(0, 0.12, ages.size)
true_risk = 5 + 2.0 * vibration + 0.35 * ages

# Build a biased model from newer machines.
train_mask = groups == "new"
slope, intercept = np.polyfit(vibration[train_mask], true_risk[train_mask], 1)

# Evaluate before and after drift.
pred_before = intercept + slope * vibration
vibration_drift = vibration + np.where(groups == "old", 0.45, 0.05)
pred_after = intercept + slope * vibration_drift

# Compute signed errors by subgroup.
error_before = pred_before - true_risk
error_after = pred_after - true_risk
summary = pd.DataFrame({"group": groups, "before": error_before, "after": error_after})

# Summarize average absolute errors.
mae_before = np.mean(np.abs(error_before))
mae_after = np.mean(np.abs(error_after))
by_group = summary.groupby("group").mean().round(2)

# Print compact evidence for learners.
print(f"Overall MAE before drift: {mae_before:.2f}")
print(f"Overall MAE after drift:  {mae_after:.2f}")
print("Mean signed error by machine group:")
print(by_group.to_string())
print("Positive error means predicted risk is too high.")

# Plot subgroup error changes.
ax = by_group.plot(kind="bar", figsize=(7, 4), rot=0)
ax.set_title("Bias can change when operating data drifts")
ax.set_ylabel("Mean signed error")
ax.axhline(0, color="black", linewidth=1)
plt.tight_layout()
plt.show()



## **3. Evidence Based Recommendations**

### **3.1. Model Evidence**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_08/Lecture_B/image_03_01.jpg?v=1783735842" width="250">



>* Link predictions to real engineering behavior
>* Support decisions beyond one performance score

>* Use metrics, errors, and visuals together
>* Check features for meaningful engineering signals

>* Link model evidence to justified actions
>* Report validation, results, and engineering meaning



In [ ]:
#@title Python Code - Model Evidence

# Model evidence connects predictions to engineering behavior.
# Small examples can show responsible recommendation logic.
# This script uses transparent calculations and plots.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create deterministic pump monitoring data.
rng = np.random.default_rng(seed=42)
hours = np.arange(1, 41, dtype=float)

# Simulate measured vibration and temperature evidence.
vibration = 0.08 * hours + rng.normal(0, 0.25, hours.size)
temperature = 68 + 0.35 * hours + rng.normal(0, 1.2, hours.size)

# Simulate measured maintenance risk score.
risk = 12 + 9 * vibration + 0.55 * temperature
risk = risk + rng.normal(0, 3.0, hours.size)

# Store evidence in a compact table.
data = pd.DataFrame({"hours": hours, "vibration": vibration})
data["temperature"] = temperature

data["risk"] = risk
assert data.shape == (40, 4)

# Fit a transparent linear model using numpy.
features = data[["vibration", "temperature"]].to_numpy()
target = data["risk"].to_numpy()

# Add an intercept column for coefficients.
ones = np.ones((features.shape[0], 1))
design = np.column_stack((ones, features))

# Solve least squares without machine learning libraries.
coefficients = np.linalg.lstsq(design, target, rcond=None)[0]
predicted = design @ coefficients

# Calculate compact evidence metrics.
errors = target - predicted
mae = np.mean(np.abs(errors))

rmse = np.sqrt(np.mean(errors ** 2))
high_risk = predicted >= 85
recommended = int(np.sum(high_risk))

# Print concise evidence for decision makers.
print("Model evidence summary for pump maintenance.")
print(f"Mean absolute error: {mae:.1f} risk points.")
print(f"Root mean squared error: {rmse:.1f} risk points.")
print(f"Vibration coefficient: {coefficients[1]:.1f} risk per in/s.")
print(f"Temperature coefficient: {coefficients[2]:.1f} risk per degF.")
print(f"Recommended inspections: {recommended} of {len(data)} records.")

# Plot measured versus predicted risk evidence.
plt.figure(figsize=(7, 4))
plt.scatter(target, predicted, color="steelblue", label="records")

# Add perfect agreement reference line.
limits = [target.min() - 5, target.max() + 5]
plt.plot(limits, limits, "k--", label="perfect agreement")

# Label the engineering evidence plot.
plt.xlabel("Measured maintenance risk")
plt.ylabel("Predicted maintenance risk")
plt.title("Model Evidence: Predicted Versus Measured Risk")
plt.legend()

# Show the single required visual result.
plt.tight_layout()
plt.show()



### **3.2. Evidence Strength**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_08/Lecture_B/image_03_02.jpg?v=1783735840" width="250">



>* Evidence must justify the engineering recommendation
>* Use multiple checks beyond headline metrics

>* Classify evidence by readiness for decisions
>* Report coverage, errors, and action consequences

>* State confidence, safeguards, and supported actions
>* Match recommendations to evidence maturity



In [ ]:
#@title Python Code - Evidence Strength

# Evidence strength connects metrics to engineering decisions.
# This example scores recommendation readiness using checks.
# Small synthetic compressor data keeps results understandable.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set deterministic values for repeatable teaching results.
rng = np.random.default_rng(42)
n = 80
hours = rng.uniform(100, 2000, n)

# Simulate compressor temperature and vibration measurements.
temp_c = 55 + 0.012 * hours + rng.normal(0, 3, n)
vib_mm_s = 1.2 + 0.0015 * hours + rng.normal(0, 0.25, n)
actual_risk = 0.03 * temp_c + 0.9 * vib_mm_s

# Create a simple transparent risk score model.
predicted_risk = 0.028 * temp_c + 0.85 * vib_mm_s + 0.15
error = predicted_risk - actual_risk
abs_error = np.abs(error)

# Validate array lengths before summarizing evidence.
assert len(abs_error) == n and len(temp_c) == n
critical_mask = actual_risk > np.percentile(actual_risk, 80)
critical_error = abs_error[critical_mask].mean()

# Build evidence checks used in responsible recommendations.
mae = abs_error.mean()
coverage = critical_mask.mean()
plausible = (0.028 > 0) and (0.85 > 0)
stable = critical_error < 0.35

# Convert checks into an evidence strength category.
checks_passed = sum([mae < 0.25, coverage >= 0.2, plausible, stable])
strength = "decision-ready" if checks_passed >= 4 else "promising"
recommendation = "pilot automated inspections" if strength == "decision-ready" else "collect more data"

# Print concise evidence summary for stakeholders.
print(f"Mean absolute error: {mae:.3f} risk units")
print(f"Critical-case error: {critical_error:.3f} risk units")
print(f"Evidence checks passed: {checks_passed} of 4")
print(f"Evidence strength: {strength}")
print(f"Recommendation: {recommendation}")

# Plot predicted versus actual risk for visual evidence.
plt.figure(figsize=(6, 4))
plt.scatter(actual_risk, predicted_risk, alpha=0.75)
plt.plot([actual_risk.min(), actual_risk.max()], [actual_risk.min(), actual_risk.max()])
plt.xlabel("Actual compressor risk score")
plt.ylabel("Predicted compressor risk score")
plt.title("Evidence Strength: Error Pattern Check")
plt.grid(True, alpha=0.3)
plt.show()



### **3.3. Limitations and Next Steps**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Mechanical Engineers/Module_08/Lecture_B/image_03_03.jpg?v=1783735838" width="250">



>* State model limits before engineering decisions
>* Explain assumptions, context, and consequences clearly

>* Link each limitation to a practical action
>* Monitor, test, and revalidate model performance

>* Match next steps to engineering risk
>* Pilot, monitor, validate, and revise decisions



In [ ]:
#@title Python Code - Limitations and Next Steps

# Evidence should guide responsible engineering recommendations.
# Limitations become useful when paired with actions.
# This example scores deployment readiness transparently.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create a small project evidence table.
items = ["test coverage", "rare failures", "explainability", "drift monitoring"]
scores = np.array([0.78, 0.35, 0.55, 0.40])
risks = np.array([0.70, 0.95, 0.80, 0.85])

# Validate matching evidence and risk lengths.
assert len(items) == len(scores) == len(risks)
assert np.all((scores >= 0) & (scores <= 1))

# Convert evidence gaps into priority scores.
gaps = 1 - scores
priority = gaps * risks

# Build a compact recommendation table.
plan = pd.DataFrame({"limitation": items, "evidence": scores, "priority": priority})
plan = plan.sort_values("priority", ascending=False)

# Print only concise decision guidance.
print("Responsible recommendation: limited pilot, not full deployment.")
print("Top next steps, ranked by risk-weighted evidence gap:")
for row in plan.head(3).itertuples(index=False):
    print(f"- Improve {row.limitation}: priority {row.priority:.2f}")

# Plot priorities for stakeholder communication.
plt.figure(figsize=(7, 4))
plt.bar(plan["limitation"], plan["priority"], color="steelblue")
plt.ylabel("Risk-weighted evidence gap")
plt.title("Limitations converted into next-step priorities")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Responsible Model Use**</font>


In this lecture, you learned to:
- Interpret model behavior using coefficients, rules, errors, plots, and feature sensitivity checks. 
- Evaluate practical limitations such as overfitting, imbalance, bias, data drift, and explainability. 
- Communicate a complete mechanical engineering AI project with evidence-based recommendations. 

<font color='yellow'>Congratulations on completing this course!</font>